# Big Mart Sales Prediction

Predict retail product sales using an XGBoost regression model.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from xgboost import XGBRegressor


## 1. Load the data

Place the original Big Mart training file at `data/Train.csv`. If it is not available, the next cell creates clearly labelled demo data so the workflow can still be tested.

In [ ]:
data_path = Path("data/Train.csv")

if data_path.exists():
    df = pd.read_csv(data_path)
    print("Loaded:", data_path)
else:
    print("Train.csv not found. Creating demo data only for testing the workflow.")
    rng = np.random.default_rng(42)
    n = 1000

    df = pd.DataFrame({
        "Item_Weight": rng.normal(12.8, 4.5, n).clip(2, 25),
        "Item_MRP": rng.uniform(30, 270, n),
        "Item_Visibility": rng.uniform(0.0, 0.25, n),
        "Item_Type": rng.choice(["Food", "Drinks", "Household"], n),
        "Outlet_Size": rng.choice(["Small", "Medium", "High"], n),
        "Outlet_Type": rng.choice(
            ["Grocery Store", "Supermarket Type1", "Supermarket Type2"], n
        ),
    })

    noise = rng.normal(0, 150, n)
    df["Item_Outlet_Sales"] = (
        df["Item_MRP"] * 7
        + df["Item_Weight"] * 20
        - df["Item_Visibility"] * 500
        + noise
    ).clip(20)

df.head()


In [ ]:
print("Shape:", df.shape)
display(df.head())
display(df.isnull().sum().sort_values(ascending=False).head(10))


## 2. Clean missing values

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns
for col in numeric_cols:
    if df[col].isna().any():
        df[col] = df[col].fillna(df[col].median())

categorical_cols = df.select_dtypes(exclude=np.number).columns
for col in categorical_cols:
    if df[col].isna().any():
        df[col] = df[col].fillna(df[col].mode()[0])

print("Remaining missing values:", int(df.isnull().sum().sum()))


## 3. Exploratory analysis

In [ ]:
df["Item_Outlet_Sales"].describe()


In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(df["Item_Outlet_Sales"], bins=30)
plt.xlabel("Item Outlet Sales")
plt.ylabel("Frequency")
plt.title("Sales Distribution")
plt.show()


## 4. Prepare features

In [ ]:
target = "Item_Outlet_Sales"
X = df.drop(columns=[target])
y = df[target]

X = pd.get_dummies(X, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("Training shape:", X_train.shape)
print("Test shape:", X_test.shape)


## 5. Train XGBoost Regressor

In [ ]:
model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42
)

model.fit(X_train, y_train)
predictions = model.predict(X_test)


## 6. Evaluate the model

In [ ]:
rmse = mean_squared_error(y_test, predictions) ** 0.5
r2 = r2_score(y_test, predictions)

print(f"RMSE: {rmse:.2f}")
print(f"R²: {r2:.4f}")


In [ ]:
plt.figure(figsize=(6, 5))
plt.scatter(y_test, predictions, alpha=0.6)
plt.xlabel("Actual Sales")
plt.ylabel("Predicted Sales")
plt.title("Actual vs Predicted Sales")
plt.show()


## 7. Conclusion

This project demonstrates an end-to-end regression workflow: data cleaning, exploratory analysis, categorical encoding, model training, and evaluation using RMSE and R².